# Label Count vs Experiment Reference

读取 `/eeg-h5-files/zarr_data/wsn/` 下所有 `label_vocab.json`，  
与 `experiment_reference.csv` 的数据集名称对齐，输出 label 数量表格。

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
ZARR_ROOT = Path("/eeg-h5-files/zarr_data/wsn")
REPO_ROOT  = Path("/benchmark-eeg/5.0_version")
REF_CSV    = REPO_ROOT / "experiment_tracking" / "experiment_reference.csv"

In [ ]:
# ── Step 1: 读取所有 label_vocab.json ─────────────────────────────────────────
zarr_records = []
for vocab_path in sorted(ZARR_ROOT.glob("*/label_vocab.json")):
    dir_name = vocab_path.parent.name
    try:
        vocab = json.loads(vocab_path.read_text(encoding="utf-8"))
        n_labels = len(vocab)
        label_names = ", ".join(v.get("name", k) for k, v in vocab.items())
    except Exception as e:
        n_labels = None
        label_names = f"ERROR: {e}"
    zarr_records.append({"zarr_dir": dir_name, "n_labels": n_labels, "label_names": label_names})

zarr_df = pd.DataFrame(zarr_records)
print(f"找到 {len(zarr_df)} 个 label_vocab.json")
zarr_df.head(10)

In [ ]:
# ── Step 2: 名称归一化，用于模糊对齐 ──────────────────────────────────────────
def normalize(name: str) -> str:
    """小写 + 去掉 _wsn/_10s 后缀 + 只保留字母数字。"""
    s = str(name).lower()
    for ch, rep in {"–": "_", "—": "_", "-": "_", " ": "_"}.items():
        s = s.replace(ch, rep)
    s = re.sub(r"_wsn$", "", s)
    s = re.sub(r"_10s$", "", s)
    s = re.sub(r"\([^)]*\)", "", s)
    return re.sub(r"[^a-z0-9]+", "", s)

zarr_df["norm"] = zarr_df["zarr_dir"].map(normalize)

# 若同一 norm key 有多个目录（如 AD65 / AD65_wsn），优先选非 _wsn 版本
zarr_df["wsn_suffix"] = zarr_df["zarr_dir"].str.endswith("_wsn")
zarr_dedup = (
    zarr_df.sort_values("wsn_suffix")
           .drop_duplicates(subset="norm", keep="first")
           .set_index("norm")
)
print(f"去重后 {len(zarr_dedup)} 条 zarr 记录")

In [ ]:
# ── Step 3: 读取 experiment_reference.csv ────────────────────────────────────
ref = pd.read_csv(REF_CSV)
print(f"experiment_reference 行数: {len(ref)}")
print("列名:", ref.columns.tolist())
ref.head()

In [ ]:
# ── Step 4: 对齐 ──────────────────────────────────────────────────────────────
# 优先用 dataset_exp_name 匹配，其次 dataset
def lookup(name: str) -> tuple:
    """返回 (zarr_dir, n_labels, label_names) 或 (None, None, None)。"""
    key = normalize(name)
    if key in zarr_dedup.index:
        row = zarr_dedup.loc[key]
        return row["zarr_dir"], row["n_labels"], row["label_names"]
    # 子串匹配兜底
    candidates = [k for k in zarr_dedup.index if key in k or k in key]
    if len(candidates) == 1:
        row = zarr_dedup.loc[candidates[0]]
        return row["zarr_dir"], row["n_labels"], row["label_names"]
    return None, None, None

rows = []
for _, r in ref.iterrows():
    exp_name = str(r.get("dataset_exp_name", "") or "")
    dataset  = str(r.get("dataset", "") or "")
    # 先尝试 exp_name，再试 dataset
    zarr_dir, n_labels, label_names = lookup(exp_name)
    if zarr_dir is None:
        zarr_dir, n_labels, label_names = lookup(dataset)
    rows.append({
        "category":         r.get("category", ""),
        "dataset_exp_name": exp_name,
        "zarr_dir":         zarr_dir or "—",
        "n_labels":         n_labels,
        "label_names":      label_names or "—",
    })

result = pd.DataFrame(rows)
print(f"匹配成功: {result['n_labels'].notna().sum()} / {len(result)}")
print(f"未匹配:   {result['n_labels'].isna().sum()} 条")

In [ ]:
# ── Step 5: 最终表格 ──────────────────────────────────────────────────────────
display(
    result[["category", "dataset_exp_name", "n_labels", "label_names"]]
    .rename(columns={
        "category":         "Category",
        "dataset_exp_name": "Dataset (exp name)",
        "n_labels":         "# Labels",
        "label_names":      "Label Names",
    })
    .reset_index(drop=True)
    .style.set_properties(**{"text-align": "left"})
)

In [ ]:
# ── 未匹配项排查 ───────────────────────────────────────────────────────────────
unmatched = result[result["n_labels"].isna()][["category", "dataset_exp_name"]]
if not unmatched.empty:
    print("以下数据集未找到对应 label_vocab.json:")
    display(unmatched.reset_index(drop=True))
else:
    print("所有数据集均已匹配！")

In [ ]:
# ── 保存结果（可选）────────────────────────────────────────────────────────────
# out_path = REPO_ROOT / "experiment_tracking" / "label_counts.csv"
# result.to_csv(out_path, index=False)
# print("已保存到", out_path)